In [1]:
import os
import sys
import matplotlib as plt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from utils.dataset import load_data, get_datasets
from utils.scores import score_summary_rouge, score_summary_bertscore

In [3]:
DATASET_PATH = os.path.join("..", "..", "data", "cleaned_datasets", "dataset_lapresse")
SUMMARIES_PATH = os.path.join("..", "..", "data", "summaries", "dataset_lapresse")

assert os.path.exists(DATASET_PATH), f"Dataset not found at {DATASET_PATH}"
assert os.path.exists(SUMMARIES_PATH) and len(os.listdir(SUMMARIES_PATH)) > 0, f"Generate summaries first (or download them)"

In [ ]:
x = load_data(DATASET_PATH, SUMMARIES_PATH)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

import re

csv_path = "summary_metrics.csv"

def count_words(text):
    words = re.findall(r'\b[a-zA-ZÀ-ÖØ-öø-ÿ]+\b', text)  # Matches words with letters only
    return len(words)

def compute_scores(args):
    """Wrapper function to compute ROUGE scores for multiprocessing"""
    initial_text, summary = args
    rouge = score_summary_rouge(initial_text, summary)  # Compute ROUGE scores
    return {
        "rouge_1": rouge["ROUGE-1"],
        "rouge_2": rouge["ROUGE-2"],
        "rouge_L": rouge["ROUGE-L"],
        "bert_score": score_summary_bertscore(initial_text, summary),
        "initial_length": count_words(initial_text),
        "summary_length": count_words(summary)
    }


# Load existing CSV if available, otherwise compute metrics
if os.path.exists(csv_path):
    print("Loading precomputed metrics from CSV...")
    df = pd.read_csv(csv_path)

else:
    # Run in parallel using ThreadPoolExecutor (faster for I/O bound tasks)
    with ThreadPoolExecutor() as executor:
        results = list(tqdm(executor.map(compute_scores, x), total=len(x), desc="Processing Summaries"))

    # Convert to DataFrame
    df = pd.DataFrame(results)

    df.to_csv(csv_path, index=False)

# Display first few rows
print(df.head())


In [ ]:
csv_path = "summary_metrics.csv"
df.to_csv(csv_path, index=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define colors for each metric
colors = {
    "rouge_1": "skyblue",
    "rouge_2": "salmon",
    "rouge_L": "lightgreen",
    "bert_score": "orchid"
}

# Create figure with 2 subplots (Histogram + Scatter plot)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

## 🔹 Histogram for ROUGE & BERTScore
for metric, color in colors.items():
    axes[0].hist(df[metric], bins=20, color=color, edgecolor='black', alpha=0.5, density=True, label=metric)

    # Add an average density line
    mean_value = np.mean(df[metric])
    axes[0].axvline(mean_value, color=color, linestyle="dashed", linewidth=2, label=f"{metric} avg")

axes[0].set_title("Score Distribution (ROUGE & BERTScore)")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Density")
axes[0].legend()


## 🔹 Scatter plot: Initial text length vs. Scores
for metric, color in colors.items():
    axes[1].scatter(df['initial_length'] / df['summary_length'], df[metric], alpha=0.5, color=color, label=metric, s=10)

    # Add an average trend line
    mean_score = np.mean(df[metric])
    axes[1].axhline(mean_score, color=color, linestyle="dashed", linewidth=2)

axes[1].set_xlabel("Initial text length / summary text lenght (lengths in words)")
axes[1].set_ylabel("Score")
axes[1].set_title("Scores vs. text length factor")
axes[1].legend()

# Adjust layout and show
plt.tight_layout()
plt.show()


In [ ]:
z = get_datasets(DATASET_PATH, SUMMARIES_PATH)
print(z)

NameError: name 'DATASET_PATH' is not defined